In [2]:
from google.colab import drive

drive.mount('/content/drive')
print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


In [4]:
!pip install -q ucimlrepo
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error

cdc_diabetes = fetch_ucirepo(id=891)
df_uci = pd.concat([cdc_diabetes.data.features, cdc_diabetes.data.targets], axis=1)

df_brfss2015 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Diabetes Binary Health Indicators BRFSS2015.csv')
df_prediction = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Diabetes Prediction Dataset.csv')

print("Preprocessing datasets")
df_prediction_encoded = pd.get_dummies(df_prediction, columns=['gender', 'smoking_history'], drop_first=True)

poly = PolynomialFeatures(degree=2, include_bias=False)
def add_poly_terms(df, cols):
    poly_features = poly.fit_transform(df[cols])
    poly_cols = poly.get_feature_names_out(cols)
    df_poly = pd.DataFrame(poly_features, columns=poly_cols, index=df.index)
    df_out = pd.concat([df.drop(columns=cols), df_poly], axis=1)
    return df_out.loc[:, ~df_out.columns.duplicated()]

df_uci_poly = add_poly_terms(df_uci, ['BMI', 'MentHlth', 'PhysHlth', 'Age'])
df_brfss2015_poly = add_poly_terms(df_brfss2015, ['BMI', 'MentHlth', 'PhysHlth', 'Age'])
df_prediction_poly = add_poly_terms(df_prediction_encoded, ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level'])

def run_regularized_models(df, target_col, dataset_name):
    X = df.drop(columns=[target_col]).astype(float)
    y = df[target_col].astype(float)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    models = {
        'Ridge': Ridge(alpha=1.0, random_state=42),
        'Lasso': Lasso(alpha=0.1, random_state=42),
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)
    }

    print(f"\n--- {dataset_name} Regularized Models ---")
    for name, model in models.items():
        model.fit(X_scaled, y)
        preds = model.predict(X_scaled)
        r2 = r2_score(y, preds)
        mse = mean_squared_error(y, preds)
        print(f"{name} -> R-squared: {r2:.4f}, MSE: {mse:.4f}")

run_regularized_models(df_uci_poly, 'Diabetes_binary', 'UCI Dataset')
run_regularized_models(df_brfss2015_poly, 'Diabetes_binary', 'BRFSS2015 Dataset')
run_regularized_models(df_prediction_poly, 'diabetes', 'Prediction Dataset')

Preprocessing datasets

--- UCI Dataset Regularized Models ---
Ridge -> R-squared: 0.1722, MSE: 0.0993
Lasso -> R-squared: 0.0028, MSE: 0.1196
ElasticNet -> R-squared: 0.1045, MSE: 0.1074

--- BRFSS2015 Dataset Regularized Models ---
Ridge -> R-squared: 0.1722, MSE: 0.0993
Lasso -> R-squared: 0.0028, MSE: 0.1196
ElasticNet -> R-squared: 0.1045, MSE: 0.1074

--- Prediction Dataset Regularized Models ---
Ridge -> R-squared: 0.5183, MSE: 0.0375
Lasso -> R-squared: 0.1907, MSE: 0.0629
ElasticNet -> R-squared: 0.3256, MSE: 0.0525
